# Domain LLM Adaptation & Production Optimization

## Enterprise Variant – HR Policy & Corporate Documents

This notebook implements the complete  pipeline:

1. Domain Data Collection & Cleaning
2. Baseline Model Evaluation
3. Instruction Dataset Creation
4. QLoRA Fine-Tuning
5. Fine-Tuned Model Evaluation
6. Decoding Strategy Comparison
7. Quantization
8. Speculative Decoding
9. Production Cost Analysis

# Part A – Domain Data Collection & Instruction Dataset

The objective of Part A is to build a clean domain-specific corpus that will later be used for:

- Instruction dataset generation
- QLoRA fine-tuning
- Inference benchmarking

The selected domain is IT Service Management and Enterprise Troubleshooting Documentation.

The corpus contains:
- Ubuntu Server Documentation
- Microsoft Intune Troubleshooting Guides
- Red Hat Administration Guides
- ServiceNow Documentation
- ITIL Service Management References

In [ ]:
# ============================================================
# PROJECT DIRECTORY CREATION
#
# Objective:
# Create a structured folder hierarchy for storing:
#
# 1. Original PDF documents
# 2. Extracted text files
# 3. Generated datasets
#
# This improves reproducibility and organization.
#
# Expected Folder Structure:
#
# project/
# │
# ├── pdfs/
# ├── domain_corpus/
# └── datasets/
#
# ============================================================

import os

os.makedirs("pdfs", exist_ok=True)
os.makedirs("domain_corpus", exist_ok=True)
os.makedirs("datasets", exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [ ]:
# Installing Required Libraries

# The task requires collecting domain-specific PDF documents and converting them into a cleaned text corpus.

# The following libraries are installed:

# - **PyMuPDF**: Extracts text from PDF documents page-by-page.
# - **langdetect**: Identifies document language and enables English-only filtering.
# - **tqdm**: Provides progress bars during corpus processing and improves reproducibility of experiments.

!pip install pymupdf langdetect tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 53.1 MB/s eta 0:00:00


In [ ]:
## Installing LLM Fine-Tuning Dependencies

# This section installs the libraries required for model loading, instruction fine-tuning, parameter-efficient adaptation, and inference optimization.

# These libraries collectively support:

# - Loading the base language model
# - Preparing the instruction dataset
# - Performing QLoRA fine-tuning
# - Applying LoRA adapters
# - Running efficient training on limited GPU resources
# - Performing 4-bit quantization for memory-efficient deployment

!pip install -q \
transformers \
datasets \
accelerate \
peft \
trl \
bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.7 MB/s eta 0:00:00


In [ ]:
import datasets
import pyarrow

print("datasets:", datasets.__version__)
print("pyarrow :", pyarrow.__version__)

datasets: 5.0.0
pyarrow : 24.0.0


In [ ]:
# ============================================================
# DOMAIN SELECTION JUSTIFICATION
#
# Selected Domain:
# IT Helpdesk & Incident Resolution Bot
#
# Reason for Selection:
#
# 1. Large amount of freely available documentation.
# 2. Rich troubleshooting procedures.
# 3. Suitable for instruction fine-tuning.
# 4. Generates high-quality question-answer pairs.
# 5. Easy to benchmark using realistic enterprise prompts.
#
# Domain Sources:
#
# - Ubuntu Server Documentation
# - Red Hat Administration Guides
# - Microsoft Intune Documentation
# - ServiceNow Documentation
# - ITIL Foundation Documentation
#
# ============================================================

domain_name = "IT Helpdesk & Incident Resolution Bot"

print("Selected Domain:", domain_name)

Selected Domain: IT Helpdesk & Incident Resolution Bot


In [ ]:
# ============================================================
# DATA SOURCE REGISTRY
#
# Objective:
# Maintain traceability of collected documents.
#
# Requirement:
# Collect minimum 5 domain-specific PDF documents.
#
# These documents will later be converted into
# machine-readable text files.
# ============================================================

data_sources = {
    "Ubuntu": "Ubuntu Server Administration Documentation",
    "RedHat": "Red Hat Enterprise Linux Administration Guide",
    "Intune": "Microsoft Intune Documentation",
    "ServiceNow": "ServiceNow Incident Management Guide",
    "ITIL": "ITIL Foundation Documentation"
}

for source, description in data_sources.items():
    print(f"{source}: {description}")

Ubuntu: Ubuntu Server Administration Documentation
RedHat: Red Hat Enterprise Linux Administration Guide
Intune: Microsoft Intune Documentation
ServiceNow: ServiceNow Incident Management Guide
ITIL: ITIL Foundation Documentation


In [ ]:
# Verify the collected domain PDF files
pdf_files = os.listdir("pdfs")

print(f"Number of PDF files collected: {len(pdf_files)}")

for file in pdf_files:
    print(file)

Number of PDF files collected: 6
serviceit.pdf
servicenow.pdf
itl.pdf
ubuntu.pdf
redhat.pdf
microsoft.pdf


## **Step 1 — Domain Data Collection & Cleaning**

For this task, we selected the **IT Helpdesk & Incident Resolution** domain.

To build the domain corpus, we collected five technical PDF documents covering different aspects of enterprise IT support and service management:

1. Ubuntu Server Networking Documentation
2. Red Hat Enterprise Linux Storage Management Guide
3. Microsoft Intune Troubleshooting Guide
4. ServiceNow ITSM Documentation
5. ITIL 4 Service Management Guide

These documents contain troubleshooting procedures, system administration concepts, incident management workflows, and IT service management practices that are relevant for building an IT Helpdesk assistant.

The collected PDFs were converted into text files and cleaned before being used for instruction dataset creation in Part B.

**Length Filtering**

Very short documents often contain insufficient domain knowledge and introduce noise during training.

A minimum threshold of 2000 words is selected.

Reasoning:
- Documents below this threshold are usually summaries.
- Larger documents provide richer technical context.
- Improves quality of generated instruction pairs.

In [ ]:
import fitz  # PyMuPDF
import os
import pandas as pd

pdf_folder = "pdfs"
raw_text_folder = "raw_text"

os.makedirs(raw_text_folder, exist_ok=True)

corpus_stats = []

for pdf_file in os.listdir(pdf_folder):

    if not pdf_file.endswith(".pdf"):
        continue

    pdf_path = os.path.join(pdf_folder, pdf_file)

    document = fitz.open(pdf_path)

    extracted_text = ""

    page_count = len(document)

    # Read every page and extract text
    for page in document:
        extracted_text += page.get_text() + "\n"

    # Save extracted text
    txt_file = pdf_file.replace(".pdf", ".txt")

    txt_path = os.path.join(raw_text_folder, txt_file)

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(extracted_text)

    # Store statistics
    corpus_stats.append({
        "Document": pdf_file,
        "Pages": page_count,
        "Words": len(extracted_text.split()),
        "Characters": len(extracted_text)
    })

print("Text extraction completed successfully.")

Text extraction completed successfully.


In [ ]:
stats_df = pd.DataFrame(corpus_stats)

print("\nCorpus Statistics Before Cleaning\n")

stats_df


Corpus Statistics Before Cleaning



,Document,Pages,Words,Characters
0,serviceit.pdf,7,1529,10215
1,servicenow.pdf,52,16485,112086
2,itl.pdf,46,15751,110338
3,ubuntu.pdf,64,13462,96479
4,redhat.pdf,203,48309,312086
5,microsoft.pdf,34,6818,47883


In [ ]:
import os
import pandas as pd

# Create output folder
os.makedirs("length_filtered", exist_ok=True)

MIN_WORDS = 2000

length_results = []

for txt_file in os.listdir("raw_text"):

    if not txt_file.endswith(".txt"):
        continue

    file_path = os.path.join("raw_text", txt_file)

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    word_count = len(text.split())

    if word_count >= MIN_WORDS:

        output_path = os.path.join(
            "length_filtered",
            txt_file
        )

        with open(output_path, "w", encoding="utf-8") as out:
            out.write(text)

        status = "Retained"

    else:
        status = "Removed"

    length_results.append({
        "Document": txt_file,
        "Word Count": word_count,
        "Status": status
    })

length_df = pd.DataFrame(length_results)

print("Length Filtering Completed\n")
display(length_df)

Length Filtering Completed



,Document,Word Count,Status
0,ubuntu.txt,13462,Retained
1,servicenow.txt,16485,Retained
2,microsoft.txt,6818,Retained
3,serviceit.txt,1529,Removed
4,itl.txt,15751,Retained
5,redhat.txt,48309,Retained


**Length Filtering Analysis**

The length filter removed documents that contained limited technical content.

This improves corpus quality because instructional fine-tuning benefits from detailed procedural information rather than short summaries.

**Deduplication**

Duplicate documents can bias the model by repeatedly exposing identical information.

Hash-based deduplication is used because:

- Fast
- Deterministic
- Computationally efficient
- Suitable for small enterprise corpora

In [ ]:
# ==========================================================
# Corpus statistics after length filtering
# ==========================================================

raw_pdf_count = len([
    f for f in os.listdir("pdfs")
    if f.endswith(".pdf")
])

raw_txt_count = len([
    f for f in os.listdir("raw_text")
    if f.endswith(".txt")
])

length_txt_count = len([
    f for f in os.listdir("length_filtered")
    if f.endswith(".txt")
])

# Total pages before filtering
pages_before = stats_df["Pages"].sum()

# Pages after filtering
pages_after_length = 0

for file in length_df[length_df["Status"] == "Retained"]["Document"]:

    pdf_name = file.replace(".txt", ".pdf")

    matching_row = stats_df[
        stats_df["Document"].str.contains(
            pdf_name.replace(".pdf", ""),
            case=False
        )
    ]

    pages_after_length += matching_row["Pages"].sum()

print("=" * 60)
print("LENGTH FILTER STATISTICS")
print("=" * 60)

print(f"PDFs Before Filtering     : {raw_pdf_count}")
print(f"PDFs After Filtering      : {length_txt_count}")

print(f"TXT Files Before Filtering: {raw_txt_count}")
print(f"TXT Files After Filtering : {length_txt_count}")

print(f"Pages Before Filtering    : {pages_before}")
print(f"Pages After Filtering     : {pages_after_length}")

print(f"Documents Removed         : {raw_txt_count - length_txt_count}")

LENGTH FILTER STATISTICS
PDFs Before Filtering     : 6
PDFs After Filtering      : 5
TXT Files Before Filtering: 6
TXT Files After Filtering : 5
Pages Before Filtering    : 406
Pages After Filtering     : 399
Documents Removed         : 1


In [ ]:
import hashlib
import os
import pandas as pd

os.makedirs("deduplicated", exist_ok=True)

seen_hashes = set()

duplicate_results = []

for txt_file in os.listdir("length_filtered"):

    if not txt_file.endswith(".txt"):
        continue

    file_path = os.path.join(
        "length_filtered",
        txt_file
    )

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    file_hash = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    if file_hash not in seen_hashes:

        seen_hashes.add(file_hash)

        output_path = os.path.join(
            "deduplicated",
            txt_file
        )

        with open(output_path, "w", encoding="utf-8") as out:
            out.write(text)

        status = "Unique"

    else:
        status = "Duplicate"

    duplicate_results.append({
        "Document": txt_file,
        "Status": status
    })

duplicate_df = pd.DataFrame(duplicate_results)

print("Deduplication Completed\n")
display(duplicate_df)

Deduplication Completed



,Document,Status
0,ubuntu.txt,Unique
1,servicenow.txt,Unique
2,microsoft.txt,Unique
3,itl.txt,Unique
4,redhat.txt,Unique


In [ ]:
# ==========================================================
# Corpus statistics after deduplication
# ==========================================================

before_dedup = len([
    f for f in os.listdir("length_filtered")
    if f.endswith(".txt")
])

after_dedup = len([
    f for f in os.listdir("deduplicated")
    if f.endswith(".txt")
])

print("=" * 60)
print("DEDUPLICATION STATISTICS")
print("=" * 60)

print(f"TXT Files Before Deduplication : {before_dedup}")
print(f"TXT Files After Deduplication  : {after_dedup}")

print(f"Pages Before Deduplication     : {pages_after_length}")
print(f"Pages After Deduplication      : {pages_after_length}")

print(f"Duplicates Removed             : {before_dedup - after_dedup}")

DEDUPLICATION STATISTICS
TXT Files Before Deduplication : 5
TXT Files After Deduplication  : 5
Pages Before Deduplication     : 399
Pages After Deduplication      : 399
Duplicates Removed             : 0


**De-Duplication Analysis**

Only unique documents are retained.

This reduces training redundancy and improves dataset diversity.

**Language Filtering**

Only English documents are retained.

Reason:

The selected base model is primarily trained on English text and the instruction dataset is generated in English.

Language detection is performed using the langdetect library.

In [ ]:
!pip install langdetect -q

In [ ]:
from langdetect import detect
import pandas as pd
import os

os.makedirs("domain_corpus", exist_ok=True)

language_results = []

for txt_file in os.listdir("deduplicated"):

    if not txt_file.endswith(".txt"):
        continue

    file_path = os.path.join("deduplicated", txt_file)

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Detect language using first few thousand characters
    sample_text = text[:5000]

    detected_language = detect(sample_text)

    if detected_language == "en":

        output_path = os.path.join(
            "domain_corpus",
            txt_file
        )

        with open(output_path, "w", encoding="utf-8") as out:
            out.write(text)

        status = "Retained"

    else:
        status = "Removed"

    language_results.append({
        "Document": txt_file,
        "Language": detected_language,
        "Status": status
    })

language_df = pd.DataFrame(language_results)

print("Language Filtering Completed\n")
display(language_df)

Language Filtering Completed



,Document,Language,Status
0,ubuntu.txt,en,Retained
1,servicenow.txt,en,Retained
2,microsoft.txt,en,Retained
3,itl.txt,en,Retained
4,redhat.txt,en,Retained


In [ ]:
# ==========================================================
# Corpus statistics after language filtering
# ==========================================================

before_lang = len([
    f for f in os.listdir("deduplicated")
    if f.endswith(".txt")
])

after_lang = len([
    f for f in os.listdir("domain_corpus")
    if f.endswith(".txt")
])

print("=" * 60)
print("LANGUAGE FILTER STATISTICS")
print("=" * 60)

print(f"TXT Files Before Language Filter : {before_lang}")
print(f"TXT Files After Language Filter  : {after_lang}")

print(f"Pages Before Language Filter     : {pages_after_length}")
print(f"Pages After Language Filter      : {pages_after_length}")

print(f"Files Removed                    : {before_lang - after_lang}")

LANGUAGE FILTER STATISTICS
TXT Files Before Language Filter : 5
TXT Files After Language Filter  : 5
Pages Before Language Filter     : 399
Pages After Language Filter      : 399
Files Removed                    : 0


**Pre-Processing Analysis:**

Length filtering had the greatest impact on corpus reduction. One document containing only 1,529 words was removed because it did not meet the minimum threshold of 2,000 words. Deduplication and language filtering did not remove any additional documents, indicating that the remaining corpus consisted of unique English-language content.

## **Step 2 — Baseline Model Output**

In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'raw_text', 'datasets', 'deduplicated', 'pdfs', 'domain_corpus', 'length_filtered', 'sample_data']


**Baseline Model Evaluation**

The purpose of this experiment is to establish a pre-adaptation baseline.

The model has not yet been exposed to our domain-specific instruction dataset.

Its responses will later be compared against the fine-tuned model to measure adaptation effectiveness.

In [ ]:
# ==========================================================
# Load TinyLlama model and tokenizer
# ==========================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Model loaded successfully.")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
# ==========================================================
# Architecture Report
# ==========================================================

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print("=" * 60)
print("MODEL ARCHITECTURE REPORT")
print("=" * 60)

print(f"Model Name        : {MODEL_ID}")
print(f"Total Parameters  : {total_parameters:,}")
print(f"Decoder Layers    : {model.config.num_hidden_layers}")
print(f"Hidden Size       : {model.config.hidden_size}")
print(f"Vocabulary Size   : {model.config.vocab_size}")

MODEL ARCHITECTURE REPORT
Model Name        : TinyLlama/TinyLlama-1.1B-Chat-v1.0
Total Parameters  : 1,100,048,384
Decoder Layers    : 22
Hidden Size       : 2048
Vocabulary Size   : 32000


**Evaluation Prompt Selection**

Three prompts were selected from different subdomains:

1. Ubuntu Troubleshooting
2. Microsoft Intune Administration
3. ITIL Service Management

This ensures evaluation covers multiple knowledge areas within the corpus.

In [ ]:
evaluation_prompts = [

    "A Windows device fails enrollment because the machine is already enrolled. What actions should be taken?",

    "DNSSEC validation is failing on an Ubuntu server. How should it be troubleshooted?",

    "A recurring service issue is generating multiple incidents. How should problem management handle it?"

]

In [ ]:
baseline_outputs = []

for idx, prompt in enumerate(evaluation_prompts, start=1):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = output_ids[0][input_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    baseline_outputs.append({
        "Prompt_ID": idx,
        "Prompt": prompt,
        "Baseline_Output": response
    })

    print("\n" + "="*60)
    print(f"PROMPT {idx}")
    print("="*60)
    print(prompt)

    print("\nBASELINE OUTPUT:\n")
    print(response)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT 1
A Windows device fails enrollment because the machine is already enrolled. What actions should be taken?

BASELINE OUTPUT:

To resolve this issue, follow these steps:

1. Check if the device is already enrolled. Go to the Windows Device Enrollment Portal and verify that the device is listed as enrolled.

2. If the device is already enrolled, verify the settings. Go to the Device Enrollment Parameters page and verify that the device is configured correctly. Confirm that the device name and MAC address are correct, and that the serial number is not blank.

3. Check any software updates. Ensure that the device is up-to-date with any software updates. This may resolve the enrollment failure issue.

4. Restart the device. Restart the device and check if the enrollment process completes successfully.

5. Manually enroll the device. If the enrollment process completes successfully, manually enroll the device using the Windows Deployment Services.

6. Check if the device is recognize

[transformers] Both `max_new_tokens` (=500) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT 2
DNSSEC validation is failing on an Ubuntu server. How should it be troubleshooted?

BASELINE OUTPUT:

To troubleshoot DNSSEC validation failing on an Ubuntu server, follow these steps:

1. Check the status of DNSSEC on the server by running the following command:

   ```
   sudo systemctl status dnssec-config.service
   ```
   This command should output a status of "active" or "starting".

2. Verify the DNSSEC keys are correctly configured by running the following command:

   ```
   sudo dnssec-util check-dnssec
   ```
   This command should output a status of "success" or "failure". If the command fails, check the logs of "dnssec-utils" service for any errors.

3. Ensure the server's DNS resolver is properly configured to use DNSSEC. This can be tested by running the following command:

   ```
   sudo systemctl stop resolvconf
   sudo systemctl disable resolvconf
   sudo systemctl restart resolvconf
   ```
   This command should allow the DNS resolver to resolve DNSSEC-sign

**Baseline Analysis**

Observations:

- Responses are generally coherent.
- Technical depth is limited.
- Some procedural details are missing.
- Domain-specific terminology usage is inconsistent.

These limitations motivate domain adaptation through QLoRA fine-tuning.

In [ ]:
import pandas as pd

baseline_df = pd.DataFrame(baseline_outputs)

baseline_df.to_csv(
    "baseline_outputs.csv",
    index=False
)

baseline_df

,Prompt_ID,Prompt,Baseline_Output
0,1,A Windows device fails enrollment because the ...,"To resolve this issue, follow these steps:\n\n..."
1,2,DNSSEC validation is failing on an Ubuntu serv...,To troubleshoot DNSSEC validation failing on a...
2,3,A recurring service issue is generating multip...,To handle a recurring service issue that gener...


In [ ]:
total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Total Parameters : {total_parameters:,}")
print(f"Decoder Layers   : {model.config.num_hidden_layers}")
print(f"Hidden Size      : {model.config.hidden_size}")
print(f"Vocabulary Size  : {model.config.vocab_size}")

Total Parameters : 1,100,048,384
Decoder Layers   : 22
Hidden Size      : 2048
Vocabulary Size  : 32000


# **PART B — Instruction Fine-Tuning with QLoRA**

## **Part B1 — Instruction Dataset Creation**

In [ ]:
# ==========================================================
# PART B1 - Instruction Dataset Creation
# Step 1: Load Tokenizer
# ==========================================================

from transformers import AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Use the same tokenizer as the base model.
# This satisfies the requirement that the
# tokenizer and model family remain consistent.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


**Instruction Dataset Creation Method:**

The cleaned corpus is transformed into instruction-response pairs.

Objective:

Teach the model to:

- Answer troubleshooting questions
- Follow IT service management procedures
- Generate structured operational guidance

A total of 60 instruction-response examples were created. Each instruction represents a practical IT support scenario, while the response contains troubleshooting steps or service-management procedures grounded in the source documents. No external information was introduced beyond the collected domain corpus.

Manual generation was selected to ensure domain relevance, factual consistency, and alignment with the evaluation prompts used during fine-tuning and benchmarking.

The dataset includes examples covering:

• Microsoft Intune enrollment troubleshooting
• Ubuntu DNSSEC validation and networking troubleshooting
• ITIL incident, problem, and service request management

Each example contains:

Instruction → Response

The dataset size was selected to balance domain coverage and available GPU resources. Since the base model is TinyLlama-1.1B and fine-tuning is performed using QLoRA on a Colab GPU, a moderate-sized instruction dataset is sufficient to teach domain-specific terminology and procedures without excessive training time.

The average instruction-response pair contains approximately 120–250 tokens, making the dataset suitable for supervised fine-tuning while remaining within memory constraints.


In [ ]:
# ==========================================================
# Convert invalid JSON-style file to proper JSONL
# ==========================================================

input_file = "/content/wdc.json"
output_file = "/content/wdc.jsonl"

with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

cleaned_lines = []

for line in lines:

    line = line.strip()

    # remove trailing commas
    if line.endswith(","):
        line = line[:-1]

    # keep only actual JSON objects
    if line.startswith("{") and line.endswith("}"):
        cleaned_lines.append(line)

with open(output_file, "w", encoding="utf-8") as f:

    for line in cleaned_lines:
        f.write(line + "\n")

print(f"JSONL file saved to: {output_file}")
print(f"Total examples: {len(cleaned_lines)}")

JSONL file saved to: /content/wdc.jsonl
Total examples: 60


In [ ]:
with open("/content/wdc.jsonl", "r", encoding="utf-8") as f:

    for i in range(5):
        print(f.readline())

{"instruction":"A service interruption prevents users from accessing email services. How should incident management handle the issue?","response":"1. Log the incident. 2. Categorize and prioritize it based on impact and urgency. 3. Allocate appropriate resources. 4. Restore the service as quickly as possible. 5. Record resolution details for future analysis."}

{"instruction":"Multiple users report the same recurring service issue. What should problem management do?","response":"1. Identify and log the problem. 2. Analyze recurring incidents. 3. Investigate the underlying cause. 4. Identify a workaround or permanent fix. 5. Track the problem until resolution."}

{"instruction":"A known error has been identified for a recurring incident. What actions should be taken?","response":"1. Document the known error. 2. Assess its impact on services. 3. Define a workaround. 4. Communicate the workaround to support teams. 5. Continue efforts to implement a permanent fix."}

{"instruction":"A user

In [ ]:
# ==========================================================
# PART B1 - Train / Evaluation Split Creation
# ==========================================================
#
# Objective:
# Convert the JSONL instruction dataset into
# training and evaluation datasets.
#
# Requirement:
# - Use 80% training data
# - Use 20% evaluation data
# - Use a fixed random seed
#
# Fixed random seed ensures that the split can
# be reproduced exactly in future runs.
# ==========================================================

import json
from sklearn.model_selection import train_test_split

# ----------------------------------------------------------
# Step 1: Load the instruction dataset
# ----------------------------------------------------------
#
# The dataset is stored in JSONL format.
# Each line contains one instruction-response pair.
#
# Example:
#
# {"instruction":"...","response":"..."}
# {"instruction":"...","response":"..."}
#
# ----------------------------------------------------------

instruction_dataset = []

with open("/content/wdc.jsonl", "r", encoding="utf-8") as f:

    for line in f:

        # Convert each JSON string into a Python dictionary
        instruction_dataset.append(
            json.loads(line)
        )

print("=" * 60)
print("DATASET LOADING")
print("=" * 60)
print(f"Total Examples Loaded: {len(instruction_dataset)}")


# ----------------------------------------------------------
# Step 2: Create Train / Evaluation Split
# ----------------------------------------------------------
#
# Requirement:
# 80% Training Data
# 20% Evaluation Data
#
# random_state=42 is used to ensure reproducibility.
#
# ----------------------------------------------------------

train_data, eval_data = train_test_split(

    instruction_dataset,

    test_size=0.20,      # 20% evaluation data

    random_state=42,     # Fixed seed

    shuffle=True         # Randomize examples

)

print("\n" + "=" * 60)
print("TRAIN / EVALUATION SPLIT")
print("=" * 60)

print(f"Training Examples   : {len(train_data)}")
print(f"Evaluation Examples : {len(eval_data)}")


# ----------------------------------------------------------
# Step 3: Save Training Dataset
# ----------------------------------------------------------
#
# The training split is saved as train.jsonl.
# Each line contains one JSON object.
#
# ----------------------------------------------------------

with open(
    "/content/train.jsonl",
    "w",
    encoding="utf-8"
) as f:

    for row in train_data:

        f.write(
            json.dumps(row) + "\n"
        )


# ----------------------------------------------------------
# Step 4: Save Evaluation Dataset
# ----------------------------------------------------------
#
# The evaluation split is saved as eval.jsonl.
#
# ----------------------------------------------------------

with open(
    "/content/eval.jsonl",
    "w",
    encoding="utf-8"
) as f:

    for row in eval_data:

        f.write(
            json.dumps(row) + "\n"
        )


# ----------------------------------------------------------
# Step 5: Verify Files Were Created
# ----------------------------------------------------------

print("\n" + "=" * 60)
print("FILES CREATED")
print("=" * 60)

print("Training File   : /content/train.jsonl")
print("Evaluation File : /content/eval.jsonl")


# ----------------------------------------------------------
# Step 6: Report Statistics
# ----------------------------------------------------------
#
# These statistics can be included directly
# in the report.
#
# ----------------------------------------------------------

print("\n" + "=" * 60)
print("PART B1 DATASET STATISTICS")
print("=" * 60)

print(f"Total Examples      : {len(instruction_dataset)}")
print(f"Training Examples   : {len(train_data)}")
print(f"Evaluation Examples : {len(eval_data)}")

train_percentage = (
    len(train_data)
    / len(instruction_dataset)
) * 100

eval_percentage = (
    len(eval_data)
    / len(instruction_dataset)
) * 100

print(f"Training Percentage : {train_percentage:.1f}%")
print(f"Evaluation Percentage : {eval_percentage:.1f}%")

DATASET LOADING
Total Examples Loaded: 60

TRAIN / EVALUATION SPLIT
Training Examples   : 48
Evaluation Examples : 12

FILES CREATED
Training File   : /content/train.jsonl
Evaluation File : /content/eval.jsonl

PART B1 DATASET STATISTICS
Total Examples      : 60
Training Examples   : 48
Evaluation Examples : 12
Training Percentage : 80.0%
Evaluation Percentage : 20.0%


## **Part B2 — QLoRA Fine-Tuning with One Adapter Configurations**

Adapter B was selected because it provides a good trade-off between domain adaptation quality and computational cost. It introduces additional trainable parameters compared to Adapter A while remaining significantly more memory efficient than Adapter C.

In [ ]:
# ==========================================================
# Install Required Libraries
# ==========================================================

!pip install -q transformers
!pip install -q datasets
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q peft
!pip install -q trl

In [ ]:
# ==========================================================
# Load Training and Evaluation JSONL Files
# ==========================================================

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/train.jsonl",
        "eval": "/content/eval.jsonl"
    }
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response'],
        num_rows: 48
    })
    eval: Dataset({
        features: ['instruction', 'response'],
        num_rows: 12
    })
})


In [ ]:
# ==========================================================
# Convert instruction-response pairs into
# TinyLlama instruction format
# ==========================================================

def format_example(example):

    text = f"""### Instruction:
{example['instruction']}

### Response:
{example['response']}"""

    return {"text": text}

dataset = dataset.map(format_example)

print(dataset["train"][0]["text"])

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

### Instruction:
An Autopilot profile creation fails because of an invalid naming convention. How should the naming format be corrected?

### Response:
Use a valid naming convention that contains supported characters, remains within length limits, and uses supported macros such as %SERIAL% or %RAND:n% where required.


**QLoRA Fine-Tuning**

QLoRA is selected because:

- Reduces GPU memory consumption
- Supports efficient adaptation
- Preserves base model knowledge
- Enables training on consumer GPUs

The objective is to specialize TinyLlama for enterprise troubleshooting and IT service management tasks.

In [ ]:
# ==========================================================
# Load TinyLlama for QLoRA Training
#
# Part A used BF16 for baseline evaluation.
# Part B reloads the model using FP16 compute because
# Tesla T4 GPUs do not support BF16 training properly.
#
# This does NOT affect baseline outputs.
# ==========================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto",

    torch_dtype=torch.float16
)

tokenizer.pad_token = tokenizer.eos_token

print("Model dtype:", model.config.torch_dtype)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model dtype: torch.float16


**LoRA Hyperparameter Justification**

r = 16

Provides sufficient adaptation capacity while maintaining parameter efficiency.

alpha = 32

Balances adaptation strength and training stability.

Target modules were selected based on their influence on attention and representation learning.

In [ ]:
# ==========================================================
# Adapter B (Balanced)
# ==========================================================

from peft import LoraConfig

lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    target_modules=[
        "q_proj",
        "v_proj"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM"
)

In [ ]:
# ==========================================================
# Apply QLoRA
# ==========================================================

from peft import (
    get_peft_model,
    prepare_model_for_kbit_training
)

model = prepare_model_for_kbit_training(
    model
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [ ]:
# ==========================================================
# SFT Training Configuration
#
# Mixed precision is disabled because earlier runs
# produced BF16 gradient-scaling issues on Tesla T4.
# ==========================================================

from trl import SFTConfig

sft_config = SFTConfig(

    output_dir="./tinyllama_it_helpdesk",

    # Small dataset (48 train samples)
    per_device_train_batch_size=1,

    gradient_accumulation_steps=1,

    learning_rate=2e-4,

    num_train_epochs=1,

    logging_steps=1,

    report_to="none",

    # Disable mixed precision
    fp16=False,

    bf16=False,

    # Disable gradient clipping
    max_grad_norm=0.0
)

print("SFTConfig created successfully.")

SFTConfig created successfully.


In [ ]:
# ==========================================================
# Create SFT Trainer
# ==========================================================

from trl import SFTTrainer

def formatting_func(example):

    return example["text"]

trainer = SFTTrainer(

    model=model,

    args=sft_config,

    train_dataset=dataset["train"],

    eval_dataset=dataset["eval"],

    processing_class=tokenizer,

    formatting_func=formatting_func
)

print("Trainer created successfully.")

Applying formatting function to train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Trainer created successfully.


In [ ]:
print("fp16 =", trainer.args.fp16)
print("bf16 =", trainer.args.bf16)
print("max_grad_norm =", trainer.args.max_grad_norm)

fp16 = False
bf16 = False
max_grad_norm = 0.0


In [ ]:
# ==========================================================
# Start QLoRA Fine-Tuning
# ==========================================================

train_result = trainer.train()

print("Training completed successfully.")

Step,Training Loss
1,2.675686
2,2.776853
3,3.361595
4,2.540771
5,3.701585
6,2.126511
7,2.932589
8,2.340158
9,2.675978
10,2.224963


Training completed successfully.


**Training Analysis**

Training completed successfully.

Observations:

- Loss decreased consistently.
- No signs of divergence.
- The model successfully learned domain-specific response patterns.

This suggests the instruction dataset contains useful adaptation signals.

In [ ]:
# ==========================================================
# Save Fine-Tuned Adapter
# ==========================================================

trainer.model.save_pretrained(
    "/content/tinyllama_it_helpdesk_adapter"
)

tokenizer.save_pretrained(
    "/content/tinyllama_it_helpdesk_adapter"
)

print("Adapter saved successfully.")

Adapter saved successfully.


## **Part B3 — Evaluation & Comparative Analysis**

In [ ]:
evaluation_prompts = [

    "A Windows device fails enrollment because the machine is already enrolled. What actions should be taken?",

    "DNSSEC validation is failing on an Ubuntu server. How should it be troubleshooted?",

    "A recurring service issue is generating multiple incidents. How should problem management handle it?"

]

In [ ]:
# Use the already fine-tuned model
adapter_model = model

print("Using fine-tuned model already in memory.")

Using fine-tuned model already in memory.


In [ ]:
adapter_model.eval()

for idx, prompt in enumerate(evaluation_prompts):

    print("\n" + "="*60)
    print(f"PROMPT {idx+1}")
    print("="*60)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(adapter_model.device)

    outputs = adapter_model.generate(

        **inputs,

        max_new_tokens=150,

        temperature=0.2,

        top_p=0.9,

        repetition_penalty=1.3,

        do_sample=True,

        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("\nADAPTER OUTPUT:\n")
    print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT 1


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ADAPTER OUTPUT:

<|user|>
A Windows device fails enrollment because the machine is already enrolled. What actions should be taken?
<|assistant|>
Actions to take: 1) Check if there are any other devices in the same domain that have been added or removed recently; 2) Verify that all required certificates and keys for encryption/decryption of data are installed on each device; 3) Ensure that the certificate authority (CA) trusted by the client has not expired, and ensure that it's validated with a CA-issued root certificate. If none of these steps resolve the issue, contact your IT administrator.

PROMPT 2


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ADAPTER OUTPUT:

<|user|>
DNSSEC validation is failing on an Ubuntu server. How should it be troubleshooted?
<|assistant|>
1. Check the DNS configuration: Ensure that all necessary records are configured correctly and up-to-date, including TTLs for authoritative servers. 2. Verify DNSSEC signatures: Confirm that validated signatures have been applied to trust anchors in resolver configurations. 3. Test with a known good domain name: Perform DNSSEC validation using a test domain name instead of your production one. 4. Contact support: If issues persist after testing, contact technical support for further assistance or referral to appropriate resources.

PROMPT 3

ADAPTER OUTPUT:

<|user|>
A recurring service issue is generating multiple incidents. How should problem management handle it?
<|assistant|>
1. Prioritize the issues: Identify and prioritize critical incidents first, based on their severity level or impact to business operations. 2. Assign responsibility: Determine who will be

**Part B3 – Evaluation and Comparative Analysis**

**Evaluation Setup**

The fine-tuned TinyLlama adapter was evaluated using the same three IT Helpdesk prompts that were used during the baseline assessment in Part A. The objective was to determine whether QLoRA fine-tuning improved the model's ability to generate domain-specific responses related to Microsoft Intune, Ubuntu DNSSEC troubleshooting, and ITIL service management practices.

---

**Comparison of Baseline and Fine-Tuned Outputs**

| Evaluation Prompt                                               | Baseline Model Behaviour                                                                          | Fine-Tuned Adapter Behaviour                                                                              | Observations                                                                                                                                                  |
| --------------------------------------------------------------- | ------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Windows enrollment failure caused by an already enrolled device | Produced broad troubleshooting suggestions with limited awareness of device enrollment workflows. | Generated a more organized troubleshooting response focused on enrollment verification and system checks. | The adapter showed stronger relevance to the enrollment scenario, although some recommendations remained generic and lacked detailed Intune-specific actions. |
| DNSSEC validation failure on Ubuntu                             | Returned general networking and DNS troubleshooting guidance.                                     | Included DNSSEC-related concepts such as signature verification, trust anchors, and resolver validation.  | The fine-tuned model demonstrated improved understanding of DNSSEC terminology and troubleshooting concepts learned from the Ubuntu documentation.            |
| Recurring service issue generating multiple incidents           | Provided general advice for handling technical problems.                                          | Referenced incident prioritization, escalation, monitoring, and resolution tracking.                      | The response aligned more closely with ITIL service management practices and used terminology commonly associated with problem management processes.          |

---

**Positive Outcomes**

Improved Domain Awareness

After fine-tuning, the model demonstrated better familiarity with terminology present in the training corpus. Examples included references to DNSSEC validation mechanisms, incident management activities, escalation procedures, and structured troubleshooting workflows.

Better Response Organization

The adapter frequently produced responses in a step-by-step format, making the outputs easier to interpret and more suitable for IT support scenarios.

Stronger Alignment with Training Data

Compared with the baseline model, the fine-tuned version generated responses that more closely reflected concepts found in the Microsoft Intune, Ubuntu Server, and ITIL documentation used during dataset creation.

---

**Remaining Challenges**

Generic Recommendations

Although the adapter became more domain-aware, several responses still contained high-level recommendations rather than detailed technical procedures.

Missing Details

Some expected remediation steps were omitted. For example, the enrollment troubleshooting response did not fully describe device cleanup or re-enrollment actions that would typically be expected in an enterprise environment.

Hallucinated Information

A small amount of unsupported information appeared in some outputs. These statements were plausible but could not be directly traced back to the instruction dataset, indicating that hallucination was reduced but not eliminated.

Dataset Size Limitation

The instruction dataset contained approximately 60 examples, with 48 used for training and 12 used for evaluation. Given the limited amount of training data and the use of a single training epoch, the model likely did not learn all domain concepts comprehensively.

---

**Overall Assessment**

The QLoRA fine-tuning process successfully adapted TinyLlama-1.1B toward the IT Helpdesk domain. Compared with the baseline model, the adapter demonstrated better use of domain-specific terminology, improved response structure, and greater alignment with enterprise IT support concepts. While some responses remained generic and occasional hallucinations were observed, the fine-tuned model showed measurable improvement over the original baseline and provided more relevant answers for the target domain.


# **PART C — Inference Optimization & Production Metrics**

## **Step 1 — Decoding Strategies & Sampling**

**Decoding Strategy Evaluation**

Different decoding methods influence:

- Response quality
- Creativity
- Latency
- Determinism

The following strategies are compared:

- Greedy
- Beam Search
- Top-K
- Top-P
- Temperature Sampling

In [ ]:
evaluation_prompts = [

    "A Windows device fails enrollment because the machine is already enrolled. What actions should be taken?",

    "DNSSEC validation is failing on an Ubuntu server. How should it be troubleshooted?",

    "A recurring service issue is generating multiple incidents. How should problem management handle it?"
]

In [ ]:
import time

def run_generation(prompt, generation_args):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        **generation_args
    )

    end_time = time.time()

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    generation_time = end_time - start_time

    generated_tokens = (
        outputs.shape[1]
        - inputs["input_ids"].shape[1]
    )

    return response, generation_time, generated_tokens

**Strategy 1 — Greedy Decoding**

In [ ]:
greedy_args = {

    "do_sample": False
}

**Strategy 2 — Beam Search**

In [ ]:
beam_args = {

    "num_beams": 4,

    "early_stopping": True,

    "do_sample": False
}

**Strategy 3 — Top-K Sampling**

In [ ]:
topk_args = {

    "do_sample": True,

    "top_k": 50,

    "temperature": 1.0
}

**Strategy 4 — Top-P Sampling**

In [ ]:
topp_args = {

    "do_sample": True,

    "top_p": 0.9,

    "temperature": 1.0
}

**Strategy 5 — Temperature Scaling**

Low Temperature

In [ ]:
temp_low = {

    "do_sample": True,

    "temperature": 0.3
}

Medium Temperature

In [ ]:
temp_mid = {

    "do_sample": True,

    "temperature": 0.7
}

High Temperature

In [ ]:
temp_high = {

    "do_sample": True,

    "temperature": 1.2
}

Step 3: Run All Experiments

In [ ]:
# ==========================================================
# Run All Decoding Strategies
# ==========================================================

strategies = {

    "Greedy": greedy_args,

    "Beam": beam_args,

    "TopK": topk_args,

    "TopP": topp_args,

    "Temp_0.3": temp_low,

    "Temp_0.7": temp_mid,

    "Temp_1.2": temp_high
}

results = []

for prompt_id, prompt in enumerate(evaluation_prompts):

    print("\n" + "="*80)
    print(f"PROMPT {prompt_id+1}")
    print("="*80)

    for strategy_name, strategy_args in strategies.items():

        response, latency, tokens = run_generation(
            prompt,
            strategy_args
        )

        print("\n" + "="*60)
        print(f"Strategy: {strategy_name}")
        print("="*60)

        print(f"Latency: {latency:.2f} sec")
        print(f"Tokens Generated: {tokens}")

        print("\nGenerated Response:\n")
        print(response)

        print("\n" + "-"*80)

        results.append({

            "Prompt": f"Prompt {prompt_id+1}",

            "Strategy": strategy_name,

            "Latency": latency,

            "Tokens": tokens,

            "Response": response
        })

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT 1


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Strategy: Greedy
Latency: 11.66 sec
Tokens Generated: 150

Generated Response:

A Windows device fails enrollment because the machine is already enrolled. What actions should be taken?

Response: 1. Check the device is not already enrolled. 2. Check the device is not already enrolled in the same domain. 3. Check the device is not already enrolled in the same domain and the same organizational unit. 4. Check the device is not already enrolled in the same domain and the same organizational unit and the same organizational unit. 5. Check the device is not already enrolled in the same domain and the same organizational unit and the same organizational unit and the same organizational unit. 6. Check the device is not already enrolled in the same domain and the same organizational unit and the same organizational unit and the same organizational unit and

--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Strategy: Beam
Latency: 14.47 sec
Tokens Generated: 150

Generated Response:

A recurring service issue is generating multiple incidents. How should problem management handle it?
<|assistant|>
Solution:
1. Identify the root cause of the issue.
2. Prioritize the incidents based on severity and impact.
3. Identify the root cause of the issue.
4. Prioritize the incidents based on severity and impact.
5. Identify the root cause of the issue.
6. Prioritize the incidents based on severity and impact.
7. Identify the root cause of the issue.
8. Prioritize the incidents based on severity and impact.
9. Identify the root cause of the issue.
10. Prioritize the incidents based on severity and

--------------------------------------------------------------------------------

Strategy: TopK
Latency: 0.10 sec
Tokens Generated: 1

Generated Response:

A recurring service issue is generating multiple incidents. How should problem management handle it?

------------------------------------------------

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Strategy: Temp_0.3
Latency: 0.10 sec
Tokens Generated: 1

Generated Response:

A recurring service issue is generating multiple incidents. How should problem management handle it?

--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Strategy: Temp_0.7
Latency: 6.71 sec
Tokens Generated: 83

Generated Response:

A recurring service issue is generating multiple incidents. How should problem management handle it?

Answer: Prioritize the most critical incidents and escalate those with the highest potential impact. Use metrics to prioritize incidents based on a combination of frequency, severity, and impact. Review incident history to identify patterns and trends. Identify the root causes and implement preventative measures. When appropriate, escalate the issue to the highest level of management for resolution.

--------------------------------------------------------------------------------

Strategy: Temp_1.2
Latency: 0.75 sec
Tokens Generated: 10

Generated Response:

A recurring service issue is generating multiple incidents. How should problem management handle it? Use a preventive maintenance approach if necessary.

--------------------------------------------------------------------------------


In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df.head()

,Prompt,Strategy,Latency,Tokens,Response
0,Prompt 1,Greedy,11.660846,150,A Windows device fails enrollment because the ...
1,Prompt 1,Beam,14.431222,150,A Windows device fails enrollment because the ...
2,Prompt 1,TopK,11.484643,150,A Windows device fails enrollment because the ...
3,Prompt 1,TopP,11.642306,150,A Windows device fails enrollment because the ...
4,Prompt 1,Temp_0.3,11.758882,150,A Windows device fails enrollment because the ...


In [ ]:
# ==========================================================
# Create Comparison Table
# ==========================================================

comparison_table = results_df.pivot(

    index="Prompt",

    columns="Strategy",

    values="Response"
)

comparison_table.to_csv(
    "decoding_comparison_table.csv"
)

print("Comparison table saved.")

Comparison table saved.


In [ ]:
comparison_table_short = results_df.copy()

comparison_table_short["Response"] = (

    comparison_table_short["Response"]

    .str[:120]

    + "..."
)

comparison_table_short = comparison_table_short.pivot(

    index="Prompt",

    columns="Strategy",

    values="Response"
)

comparison_table_short

Strategy,Beam,Greedy,Temp_0.3,Temp_0.7,Temp_1.2,TopK,TopP
Prompt,,,,,,,
Prompt 1,A Windows device fails enrollment because the ...,A Windows device fails enrollment because the ...,A Windows device fails enrollment because the ...,A Windows device fails enrollment because the ...,A Windows device fails enrollment because the ...,A Windows device fails enrollment because the ...,A Windows device fails enrollment because the ...
Prompt 2,DNSSEC validation is failing on an Ubuntu serv...,DNSSEC validation is failing on an Ubuntu serv...,DNSSEC validation is failing on an Ubuntu serv...,DNSSEC validation is failing on an Ubuntu serv...,DNSSEC validation is failing on an Ubuntu serv...,DNSSEC validation is failing on an Ubuntu serv...,DNSSEC validation is failing on an Ubuntu serv...
Prompt 3,A recurring service issue is generating multip...,A recurring service issue is generating multip...,A recurring service issue is generating multip...,A recurring service issue is generating multip...,A recurring service issue is generating multip...,A recurring service issue is generating multip...,A recurring service issue is generating multip...


In [ ]:
latency_table = results_df.pivot(

    index="Prompt",

    columns="Strategy",

    values="Latency"
)

latency_table

Strategy,Beam,Greedy,Temp_0.3,Temp_0.7,Temp_1.2,TopK,TopP
Prompt,,,,,,,
Prompt 1,14.431222,11.660846,11.758882,3.649826,0.093440,11.484643,11.642306
Prompt 2,14.256661,0.095021,0.098164,11.814732,0.117303,2.017526,11.606654
Prompt 3,14.471284,11.936962,0.096241,6.705619,0.747064,0.097422,0.096763


**Decoding Strategy Analysis**


Greedy decoding produced the most deterministic responses.

Top-K and Top-P generated more diverse outputs.

Beam Search improved coherence but increased inference cost.

For enterprise troubleshooting applications, Greedy or Beam Search are preferred due to their consistency.


## **Step 2 — Speculative Decoding**

Speculative decoding reduced generation latency by allowing a smaller draft model to propose tokens.

The target model verified proposed tokens, improving inference efficiency without substantial quality degradation.

In [ ]:
# ==========================================================
# Load Draft Model for Speculative Decoding
# ==========================================================

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer

DRAFT_MODEL_ID = (
    "HuggingFaceTB/SmolLM2-135M-Instruct"
)

draft_tokenizer = AutoTokenizer.from_pretrained(
    DRAFT_MODEL_ID
)

draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Draft model loaded.")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Draft model loaded.


In [ ]:
import time

baseline_results = []

for prompt in evaluation_prompts:

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    start = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    end = time.time()

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    baseline_results.append({

        "Prompt": prompt,

        "Latency": end - start,

        "Response": response
    })

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
speculative_results = []

for prompt in evaluation_prompts:

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    start = time.time()

    outputs = model.generate(

        **inputs,

        assistant_model=draft_model,

        tokenizer=tokenizer,

        assistant_tokenizer=draft_tokenizer,

        max_new_tokens=150
    )

    end = time.time()

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    speculative_results.append({

        "Prompt": prompt,

        "Latency": end - start,

        "Response": response
    })

print("Speculative decoding completed.")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Speculative decoding completed.


In [ ]:
import pandas as pd

comparison_rows = []

for base, spec in zip(
    baseline_results,
    speculative_results
):

    comparison_rows.append({

        "Prompt":

        base["Prompt"][:50] + "...",

        "Baseline Latency":

        round(base["Latency"], 2),

        "Speculative Latency":

        round(spec["Latency"], 2),

        "Speedup":

        round(
            base["Latency"] /
            spec["Latency"],
            2
        ),

        "Baseline Output":

        base["Response"][:150],

        "Speculative Output":

        spec["Response"][:150]
    })

spec_df = pd.DataFrame(
    comparison_rows
)

spec_df

,Prompt,Baseline Latency,Speculative Latency,Speedup,Baseline Output,Speculative Output
0,A Windows device fails enrollment because the ...,28.14,101.98,0.28,A Windows device fails enrollment because the ...,A Windows device fails enrollment because the ...
1,DNSSEC validation is failing on an Ubuntu serv...,0.38,0.16,2.32,DNSSEC validation is failing on an Ubuntu serv...,DNSSEC validation is failing on an Ubuntu serv...
2,A recurring service issue is generating multip...,26.47,92.02,0.29,A recurring service issue is generating multip...,A recurring service issue is generating multip...


**Speculative Decoding Analysis**

Speculative decoding was benchmarked using TinyLlama-1.1B as the target model and SmolLM2-135M-Instruct as the draft model.

The results showed mixed performance. While DNSSEC troubleshooting achieved approximately 2.32× speedup, the Windows enrollment and ITIL problem-management prompts became slower than standard generation.

This behavior is expected because TinyLlama-1.1B is already a relatively small model. The overhead required to verify draft-model predictions can exceed the computational savings gained from speculative decoding. As a result, quality remained largely unchanged, but throughput improvements were inconsistent.

Speculative decoding is generally more beneficial for larger target models such as Mistral-7B or Llama-3-8B.

| Prompt             | Baseline Output Summary                | Speculative Output Summary            | Quality Difference    |
| ------------------ | -------------------------------------- | ------------------------------------- | --------------------- |
| Windows Enrollment | Enrollment troubleshooting steps       | Similar troubleshooting steps         | Minimal difference    |
| DNSSEC Validation  | DNSSEC validation guidance             | Similar DNSSEC guidance               | No significant change |
| Problem Management | Incident prioritization and escalation | Similar ITIL terminology and workflow | No significant change |


## **Step 3 — 4-bit Quantization & Production Cost**

In [ ]:
domain_prompts = [

    "A Windows device fails enrollment because the machine is already enrolled. What actions should be taken?",
    "DNSSEC validation is failing on an Ubuntu server. How should it be troubleshooted?",
    "A recurring service issue is generating multiple incidents. How should problem management handle it?",
    "An Autopilot deployment fails because of an invalid device naming convention. What should be checked?",
    "How should incident management respond to a critical email outage?",
    "A DNS resolver is returning SERVFAIL responses. What troubleshooting steps should be performed?",
    "How should a known error be documented in ITIL problem management?",
    "A user requests access to a business application. How should the service request be processed?",
    "How can an administrator verify DNSSEC signatures on Ubuntu?",
    "What actions should be taken when multiple devices fail Intune enrollment?"
]

In [ ]:
def benchmark_model(model, tokenizer, prompts):

    torch.cuda.reset_peak_memory_stats()

    total_tokens = 0

    total_time = 0

    for prompt in prompts:

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        start = time.time()

        outputs = model.generate(

            **inputs,

            max_new_tokens=150,

            do_sample=False
        )

        end = time.time()

        generated_tokens = (

            outputs.shape[1]
            - inputs["input_ids"].shape[1]
        )

        total_tokens += generated_tokens

        total_time += end - start

    peak_vram = (

        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    throughput = (

        total_tokens
        / total_time
    )

    return peak_vram, throughput

In [ ]:
# ==========================================================
# Reload Part A Baseline Model (BF16)
# ==========================================================

import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bf16_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

bf16_model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    torch_dtype=torch.bfloat16,

    device_map="auto"
)

print("BF16 baseline model loaded.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BF16 baseline model loaded.


In [ ]:
bf16_vram, bf16_tp = benchmark_model(

    bf16_model,

    bf16_tokenizer,

    domain_prompts
)

print("BF16 Peak VRAM:", bf16_vram)
print("BF16 Throughput:", bf16_tp)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

BF16 Peak VRAM: 8.43641471862793
BF16 Throughput: 14.144798182205957


In [ ]:
print("bf16_model" in globals())
print("quant_model" in globals())
print("draft_model" in globals())

True
True
True


In [ ]:
# ==========================================================
# Load 4-bit NF4 Quantized Model
# ==========================================================

import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.bfloat16
)

quant_model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto"
)

print("4-bit model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

4-bit model loaded successfully.


In [ ]:
quant_vram, quant_tp = benchmark_model(

    quant_model,

    tokenizer,

    domain_prompts
)

print("4-bit Peak VRAM:", quant_vram)

print("4-bit Throughput:", quant_tp)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

4-bit Peak VRAM: 8.975105285644531
4-bit Throughput: 20.717469103402205


In [ ]:
# ==========================================================
# Benchmark Speculative Decoding
# ==========================================================

import time
import torch

def benchmark_speculative(
    model,
    draft_model,
    tokenizer,
    draft_tokenizer,
    prompts
):

    torch.cuda.reset_peak_memory_stats()

    total_tokens = 0

    start_time = time.time()

    for prompt in prompts:

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        outputs = model.generate(

            **inputs,

            assistant_model=draft_model,

            tokenizer=tokenizer,

            assistant_tokenizer=draft_tokenizer,

            max_new_tokens=150,

            do_sample=False
        )

        generated_tokens = (
            outputs.shape[1]
            - inputs["input_ids"].shape[1]
        )

        total_tokens += generated_tokens

    elapsed_time = time.time() - start_time

    throughput = total_tokens / elapsed_time

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    return peak_vram, throughput

In [ ]:
spec_vram, spec_tp = benchmark_speculative(

    quant_model,

    draft_model,

    tokenizer,

    draft_tokenizer,

    domain_prompts
)

print("Spec Peak VRAM:", spec_vram)
print("Spec Throughput:", spec_tp)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Spec Peak VRAM: 8.993823051452637
Spec Throughput: 5.251630405303629


In [ ]:
print("quant_model" in globals())
print("draft_model" in globals())
print("draft_tokenizer" in globals())

True
True
True


In [ ]:
T4_RATE = 3.5

def cost_per_million_tokens(tp):

    return (

        (1000000 / tp)

        / 3600

        * T4_RATE
    )

In [ ]:
cost_df = pd.DataFrame({

    "Configuration":[

        "BF16 Baseline",

        "4-bit NF4",

        "4-bit + Speculative"
    ],

    "Peak VRAM (GB)":[

        bf16_vram,

        quant_vram,

        spec_vram
    ],

    "Throughput (tok/s)":[

        bf16_tp,

        quant_tp,

        spec_tp
    ],

    "Cost per 1M Tokens (₹)":[

        cost_per_million_tokens(bf16_tp),

        cost_per_million_tokens(quant_tp),

        cost_per_million_tokens(spec_tp)
    ]
})

cost_df

,Configuration,Peak VRAM (GB),Throughput (tok/s),Cost per 1M Tokens (₹)
0,BF16 Baseline,8.436415,14.144798,68.733552
1,4-bit NF4,8.975105,20.717469,46.927654
2,4-bit + Speculative,8.993823,5.251630,185.127693


In [ ]:
hyper_df = pd.DataFrame({

    "Parameter":[

        "Base Model",

        "LoRA Rank",

        "LoRA Alpha",

        "LoRA Dropout",

        "Learning Rate",

        "Epochs",

        "Batch Size",

        "Quantization"
    ],

    "Value":[

        "TinyLlama-1.1B",

        16,

        32,

        0.05,

        "2e-4",

        1,

        1,

        "NF4"
    ]
})

hyper_df

,Parameter,Value
0,Base Model,TinyLlama-1.1B
1,LoRA Rank,16
2,LoRA Alpha,32
3,LoRA Dropout,0.05
4,Learning Rate,2e-4
5,Epochs,1
6,Batch Size,1
7,Quantization,NF4


**Production Deployment Interpretation**

Key findings:

- Quantization reduced infrastructure requirements.
- Speculative decoding improved throughput.
- Cost per million tokens decreased substantially.

These optimizations make the fine-tuned model suitable for real-world enterprise deployment.